In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
load_dotenv()

True

In [8]:
loader=PyPDFLoader("data_test/hr_policy_manual.pdf")
pages = loader.load()


splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_size=512,
    chunk_overlap=100,
)
chunks = splitter.split_documents(pages)


embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)


In [9]:
ret = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [13]:
ret.invoke("what is the notice period for senior managers?")

[Document(id='8a485793-fae3-425c-86c5-3e73bb7d13a3', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-31T19:53:57+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-31T19:53:57+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data_test/hr_policy_manual.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='manager and HR Business Partner.\n1.2 Notice Period\nPost-confirmation, the standard notice period is 60 days for all employees up to Senior Manager\nlevel. For Associate Director and above, the notice period is 90 days. Notice period buyout is\npermitted at basic salary per day, subject to management approval.\n1.3 Working Hours\nStandard working hours are 9 hours per day, Monday to Friday. Core hours are 10:00 AM to 5:00\nPM IST. Flexible start is permitted between 8:00 AM and 10:30 AM. Work from Home (WFH) is'),
 Document(id='0f102377-2bf9-4d

In [16]:
prompt = ChatPromptTemplate.from_template(
       "Answer using context only.\nContext: {context}\nQuestion: {question}"
)

def fmt(docs):
    return "\n\n".join([doc.page_content for doc in docs])


chain = (
    {"context": ret | fmt, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

# chain.invoke("what is the notice period for senior managers?")

In [11]:
def ask(q):
    return chain.invoke(q), ret.invoke(q)

In [17]:
ans, src = ask("What is the notice period for senior managers?")

In [18]:
ans

'The notice period for senior managers is 60 days.'

In [21]:
for i, d in enumerate(src,1):
    print("###")
    print(f"{i}. {d.page_content}")

###
1. manager and HR Business Partner.
1.2 Notice Period
Post-confirmation, the standard notice period is 60 days for all employees up to Senior Manager
level. For Associate Director and above, the notice period is 90 days. Notice period buyout is
permitted at basic salary per day, subject to management approval.
1.3 Working Hours
Standard working hours are 9 hours per day, Monday to Friday. Core hours are 10:00 AM to 5:00
PM IST. Flexible start is permitted between 8:00 AM and 10:30 AM. Work from Home (WFH) is
###
2. NovaTech Solutions Pvt. Ltd.
Employee HR Policy Manual — FY 2024-25 | Version 3.2 | Confidential
Section 1: Employment Terms
1.1 Probation Period
All new employees at NovaTech Solutions are subject to a probation period of 6 months from their
date of joining. During the probation period, either party may terminate employment with a notice
period of 15 days. Confirmation is subject to a satisfactory performance review by the reporting
manager and HR Business Partner.
1.2 

In [23]:
ans2, srcs2 = ask("please summarize the entire documents")

print(ans2)
for i, d in enumerate(srcs2,1):
    print("###")
    print(f"{i}. {d.page_content}")


The document outlines policies related to paternity leave, confidentiality, data security, and disciplinary actions. It specifies that complaints regarding bullying and discrimination must be submitted to the Internal Complaints Committee (ICC) within three months, with inquiries completed within 90 days. Employees are required to sign a Non-Disclosure Agreement (NDA) before joining, and sharing confidential data externally without legal approval can lead to termination and legal consequences. The current policy supersedes all previous HR policies, and queries can be directed to hr@novatech.in or through the HR portal.
###
1. to documentation.
2.4 Paternity Leave
###
2. bullying, and discrimination. Complaints must be submitted to the Internal Complaints Committee
(ICC) within 3 months of the incident. ICC completes inquiry within 90 days.
4.2 Confidentiality and Data Security
All employees must sign the NDA before joining. Confidential data must not be shared externally
without writte

In [29]:
# load all 3 documebnts into vs
files =["data_test/hr_policy_manual.pdf",
        "data_test/novacrm_product_manual.pdf",
        "data_test/annual_financial_report_2024.pdf"]
all_pages =[]
for pdf in files:
    loader=PyPDFLoader(pdf)
    pages = loader.load()
    all_pages.extend(pages)
    
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_size=512,
    chunk_overlap=100,
)
chunks = splitter.split_documents(all_pages)
vectorstore = FAISS.from_documents(chunks, embeddings)

ret_all    = vectorstore.as_retriever(search_kwargs={"k": 5})

chain_all = (
    {"context": ret_all | fmt, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [30]:
q = "What was the company's performance last year and what are employee benefits?"
ans4  = chain_all.invoke(q)
srcs4 = ret_all.invoke(q)

In [32]:
print(ans4)

In the financial year 2022-23, NovaTech Solutions reported a total revenue of Rs. 221 Crore, a gross profit of Rs. 103 Crore, EBITDA of Rs. 42 Crore, and a net profit of Rs. 24 Crore. The EBITDA margin was 19.0%, and the net profit margin was 10.9%. The employee count at the end of the year was 1,391.

Regarding employee benefits, the salary structure includes components such as Basic Salary (40% of CTC, fully taxable), House Rent Allowance (20%, partially exempt), Special Allowance (20%, fully taxable), Performance Bonus (10%, fully taxable), Provident Fund (5%, exempt up to limits), Gratuity (3%, exempt up to Rs. 20 lakhs), and Medical Reimbursement (2%, exempt up to Rs. 15,000 p.a.). Additionally, NovaTech contributes 12% of the Basic Salary towards the Employee Provident Fund (EPF). Appraisals are conducted annually in April, with salary revisions communicated by March 31. Mid-year promotions may be considered in October for exceptional performers.


In [33]:
for d in srcs4:
    print("###")
    print(d.metadata['source'].split("/")[-1])
    print(d.page_content)
    print("###")

###
hr_policy_manual.pdf
Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Appraisal Cycle
Appraisals are conducted annually in April. Process: self-assessment, manager rating, calibration.
Salary revisions effective 1st April are communicated by March 31. Mid-year promotions may be
considered in October for exceptional performers.
3.3 Employee Provident Fund (EPF)
NovaTech contributes 12% of Basic Salary towards EPF per the EPF Act, 1952. Employee
###
###
hr_policy_manual.pdf
Section 3: Compensation and Benefits
3.1 Salary Structure
The salary structure at NovaTech comprises the following components:
Component
% of CTC
Tax Treatment
Basic Salary
40%
Fully Taxable
House Rent Allowance (HRA)
20%
Partially Exempt (Sec 10(13A))
Special Allowance
20%
Fully Taxable
Performance Bonus
10%
Fully Taxable
Provident Fund (Employer)
5%
Exempt up to limits
Gratuity (Employer)
3%
Exempt up to Rs. 20 lakhs
Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Appra

In [34]:
q2   = "Give me a summary of all three documents."
ans5 = chain_all.invoke(q2)
srcs5 = ret_all.invoke(q2)


In [36]:
print(ans5)

The documents cover various aspects of company policies at NovaTech. 

1. **Pricing and Proposals**: NovaTech supports both GST-inclusive and GST-exclusive pricing, offers multi-currency quotes (INR, USD, AED, SGD), and allows for product bundling and volume-based discounts. Proposals can be generated as branded PDFs using configurable templates and can be signed electronically via Aadhaar eSign and DocuSign.

2. **Compensation and Benefits**: The salary structure consists of several components, including Basic Salary (40% of CTC, fully taxable), House Rent Allowance (20%, partially exempt), Special Allowance (20%, fully taxable), Performance Bonus (10%, fully taxable), Provident Fund (5%, exempt up to limits), Gratuity (3%, exempt up to Rs. 20 lakhs), and Medical Reimbursement (2%, exempt up to Rs. 15,000 p.a.). There is also a section on the performance appraisal cycle.

3. **Workplace Conduct and Security**: The policy addresses workplace conduct, stating that complaints regarding b

In [37]:
for d in srcs5:
    print("###")
    print(d.metadata['source'].split("/")[-1])
    print(d.page_content)
    print("###")

###
novacrm_product_manual.pdf
Supports GST-inclusive and GST-exclusive pricing, multi-currency quotes (INR, USD, AED, SGD),
product bundling, and volume-based discount slabs. Proposals generated as branded PDFs from
configurable templates. E-signature via Aadhaar eSign and DocuSign.
###
###
hr_policy_manual.pdf
to documentation.
2.4 Paternity Leave
###
###
hr_policy_manual.pdf
Section 3: Compensation and Benefits
3.1 Salary Structure
The salary structure at NovaTech comprises the following components:
Component
% of CTC
Tax Treatment
Basic Salary
40%
Fully Taxable
House Rent Allowance (HRA)
20%
Partially Exempt (Sec 10(13A))
Special Allowance
20%
Fully Taxable
Performance Bonus
10%
Fully Taxable
Provident Fund (Employer)
5%
Exempt up to limits
Gratuity (Employer)
3%
Exempt up to Rs. 20 lakhs
Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Appraisal Cycle
###
###
hr_policy_manual.pdf
bullying, and discrimination. Complaints must be submitted to the Internal Compla